# SmartBite YOLO26s-OBB D2S Product Crop Fine-Tuning

Fine-tunes Ultralytics `yolo26s-obb.pt` on the D2S single-class product-crop OBB dataset converted from COCO masks.

Flow:
1. Mount Drive.
2. Unzip `d2s-product-obb-yolo26s.zip`.
3. Patch `data.yaml` for Colab paths.
4. Train from generic YOLO26s-OBB weights.
5. Validate, preview, and save artifacts back to Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile


def run_live(cmd, env=None, cwd=None):
    printable = ' '.join(str(part) for part in cmd)
    print('>>', printable, flush=True)
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'Command failed with exit code {code}: {printable}')


In [ ]:
DATASET_ZIP = Path('/content/drive/My Drive/sb-colab/d2s-product-obb-yolo26s.zip')
BASE_UNZIP_DIR = Path('/content/yolo_obb_dataset')
LOCAL_DATASET_ROOT = BASE_UNZIP_DIR / 'd2s-product-obb-yolo26s'

YOLO26S_OBB_PT_DRIVE = Path('/content/drive/My Drive/sb-colab/models/yolo26s-obb.pt')
YOLO_MODEL_SOURCE = str(YOLO26S_OBB_PT_DRIVE) if YOLO26S_OBB_PT_DRIVE.exists() else 'yolo26s-obb.pt'

RUNS_PROJECT = Path('/content/output')
RUN_NAME = 'smartbite_yolo26s_obb_d2s_product_crop'
FINAL_MODEL_DRIVE_DIR = Path('/content/drive/My Drive/sb-colab/smartbite_yolo26s_obb_d2s_product_crop')
FINAL_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/smartbite_yolo26s_obb_d2s_product_crop.zip')

EPOCHS = 80
IMGSZ = 1024
BATCH = 12
DEVICE = '0'
WORKERS = 2
PATIENCE = 20
MAX_DET = 80

assert DATASET_ZIP.exists(), f'Missing dataset zip: {DATASET_ZIP}'
print('DATASET_ZIP =', DATASET_ZIP)
print('YOLO_MODEL_SOURCE =', YOLO_MODEL_SOURCE)
print('RUN_NAME =', RUN_NAME)
print('FINAL_MODEL_DRIVE_DIR =', FINAL_MODEL_DRIVE_DIR)


In [ ]:
if LOCAL_DATASET_ROOT.exists():
    shutil.rmtree(LOCAL_DATASET_ROOT)
BASE_UNZIP_DIR.mkdir(parents=True, exist_ok=True)
run_live(['unzip', '-q', '-o', DATASET_ZIP, '-d', BASE_UNZIP_DIR])

candidates = []
for yaml_path in BASE_UNZIP_DIR.rglob('data.yaml'):
    root = yaml_path.parent
    if (root / 'train' / 'images').exists() and (root / 'valid' / 'images').exists():
        candidates.append(root)
assert candidates, 'Could not find YOLO OBB dataset root containing data.yaml and train/images.'
YOLO_DATASET_ROOT = sorted(candidates, key=lambda p: len(str(p)))[0]
DATASET_YAML = YOLO_DATASET_ROOT / 'data.yaml'
print('YOLO_DATASET_ROOT =', YOLO_DATASET_ROOT)
print('DATASET_YAML =', DATASET_YAML)


In [ ]:
import yaml

cfg = yaml.safe_load(DATASET_YAML.read_text())
cfg['path'] = str(YOLO_DATASET_ROOT)
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'
cfg['names'] = {0: 'product'}
DATASET_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(DATASET_YAML.read_text())

for split in ['train', 'valid', 'test']:
    image_count = len(list((YOLO_DATASET_ROOT / split / 'images').glob('*')))
    label_count = len(list((YOLO_DATASET_ROOT / split / 'labels').glob('*.txt')))
    print(f'{split}: images={image_count} labels={label_count}')


In [ ]:
run_live([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'pyyaml'])
run_live(['nvidia-smi'])

from ultralytics import YOLO
probe = YOLO(YOLO_MODEL_SOURCE)
print('Loaded model:', YOLO_MODEL_SOURCE)
print('Task:', getattr(probe, 'task', None))


In [ ]:
train_script = f'''
import contextlib
import os
import sys
from ultralytics import YOLO

PRINT_EVERY = 50
state = {{"batch": 0}}


def log(msg):
    print(msg, file=sys.stderr, flush=True)


def on_train_batch_end(trainer):
    state["batch"] += 1
    if state["batch"] % PRINT_EVERY == 0:
        epoch = getattr(trainer, "epoch", 0) + 1
        epochs = getattr(trainer, "epochs", "?")
        loss_items = getattr(trainer, "loss_items", None)
        log(f"epoch {{epoch}}/{{epochs}} step {{state['batch']}} loss={{loss_items}}")


def on_fit_epoch_end(trainer):
    epoch = getattr(trainer, "epoch", 0) + 1
    metrics = getattr(trainer, "metrics", None)
    log(f"epoch {{epoch}} finished metrics={{metrics}}")


model = YOLO(r"{YOLO_MODEL_SOURCE}")
model.add_callback("on_train_batch_end", on_train_batch_end)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
    results = model.train(
        data=r"{DATASET_YAML}",
        epochs={EPOCHS},
        imgsz={IMGSZ},
        batch={BATCH},
        device=r"{DEVICE}",
        project=r"{RUNS_PROJECT}",
        name=r"{RUN_NAME}",
        workers={WORKERS},
        patience={PATIENCE},
        cache=False,
        plots=True,
        close_mosaic=10,
        max_det={MAX_DET},
        verbose=False,
    )

log(results)
log("Training finished.")
'''

run_live([sys.executable, '-c', train_script])


In [ ]:
best_pt = RUNS_PROJECT / RUN_NAME / 'weights' / 'best.pt'
last_pt = RUNS_PROJECT / RUN_NAME / 'weights' / 'last.pt'
assert best_pt.exists(), f'Missing best checkpoint: {best_pt}'
print('best_pt =', best_pt)
print('last_pt =', last_pt, last_pt.exists())


In [ ]:
val_script = f'''
import json
from ultralytics import YOLO


def compact_metrics(metrics):
    return {{k: float(v) for k, v in metrics.results_dict.items()}}

model = YOLO(r"{best_pt}")
print('--- VAL ---')
metrics_val = model.val(data=r"{DATASET_YAML}", imgsz={IMGSZ}, batch={BATCH}, device=r"{DEVICE}", split='val')
print(json.dumps(compact_metrics(metrics_val), indent=2))
print('--- TEST ---')
metrics_test = model.val(data=r"{DATASET_YAML}", imgsz={IMGSZ}, batch={BATCH}, device=r"{DEVICE}", split='test')
print(json.dumps(compact_metrics(metrics_test), indent=2))
'''
run_live([sys.executable, '-c', val_script])


In [ ]:
preview_script = f'''
from pathlib import Path
from ultralytics import YOLO

model = YOLO(r"{best_pt}")
source = Path(r"{YOLO_DATASET_ROOT}") / 'test' / 'images'
results = model.predict(
    source=str(source),
    imgsz={IMGSZ},
    conf=0.05,
    device=r"{DEVICE}",
    project=r"{RUNS_PROJECT}",
    name=r"{RUN_NAME}_preview",
    save=True,
    max_det={MAX_DET},
)
print('Preview images saved to:', Path(r"{RUNS_PROJECT}") / '{RUN_NAME}_preview')
print('Predicted images:', len(results))
'''
run_live([sys.executable, '-c', preview_script])


In [ ]:
src = RUNS_PROJECT / RUN_NAME
assert src.exists(), f'Missing run directory: {src}'
if FINAL_MODEL_DRIVE_DIR.exists():
    shutil.rmtree(FINAL_MODEL_DRIVE_DIR)
FINAL_MODEL_DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, FINAL_MODEL_DRIVE_DIR)
print('Saved YOLO26s-OBB artifacts to:', FINAL_MODEL_DRIVE_DIR)

if FINAL_ZIP_DRIVE.exists():
    FINAL_ZIP_DRIVE.unlink()
with zipfile.ZipFile(FINAL_ZIP_DRIVE, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(FINAL_MODEL_DRIVE_DIR.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(FINAL_MODEL_DRIVE_DIR.parent))
print('Saved artifact zip to:', FINAL_ZIP_DRIVE)
print('Best checkpoint:', FINAL_MODEL_DRIVE_DIR / 'weights' / 'best.pt')


## Notes

- If Colab runs out of memory at `BATCH = 12`, lower it to `8` or `4` and rerun.
- The dataset is single-class `product`; all original D2S categories are intentionally collapsed.
- The notebook starts from generic `yolo26s-obb.pt`, not an expiry-date detector checkpoint.
